# Build a MOM5 Basin Mask (ACCESS-OM2 style)

This notebook builds basin masks similar to `lsmask_ACCESS-OM2_1deg_20110618.nc`.

It follows this workflow:
1. Load your MOM5 grid/mask dataset.
2. Build 2D basin IDs on each native MOM5 grid location (T, UU, UT, TU).
3. Expand to 3D using vertical wet masks (`kmt`/`kmu`) when available.
4. Write a NetCDF with `mask_ttcell`, `mask_uucell`, `mask_utcell`, `mask_tucell`.

Important note: use the native MOM5 grid locations for classification. The supergrid is useful for geometry and bounds, but basin membership should be computed on the actual target grid cells.

In [ ]:
# If needed, install extra package once in this kernel.
# Uncomment if regionmask is missing:
# %pip install regionmask

In [ ]:
import numpy as np
import xarray as xr

from access_moppy.ocean_supergrid import Supergrid

try:
    import regionmask
except Exception as exc:
    raise ImportError("regionmask is required. Install with: pip install regionmask") from exc

## 1) Configure input and output paths

Set `GRID_FILE` to a file containing MOM5 coordinates and wet masks (or `kmt`/`kmu`).

Typical coordinate names used below:
- T-grid: `xt_ocean`, `yt_ocean`, `geolon_t`, `geolat_t`
- C/UU-grid: `xu_ocean`, `yu_ocean`, `geolon_c`, `geolat_c`

In [ ]:
GRID_FILE = "/path/to/your/ocean_grid_or_static_file.nc"
OUTPUT_FILE = "./lsmask_ACCESS-OM2_1deg_generated.nc"

# Optional: derive geolon/geolat from ACCESS-MOPPy supergrid helper.
# Use this when your input file has masks/kmt but not complete geolon/geolat fields.
USE_MOPPY_SUPERGRID = False
NOMINAL_RESOLUTION = "100 km"  # one of: "100 km", "25 km", "10 km"

# Number of vertical levels in output if st_ocean is not present in input
DEFAULT_NZ = 50

In [ ]:
ds = xr.open_dataset(GRID_FILE)

if USE_MOPPY_SUPERGRID:
    sg = Supergrid(NOMINAL_RESOLUTION)
    grid_t = sg.extract_grid("T", "B")
    grid_c = sg.extract_grid("C", "B")

    ny_t, nx_t = grid_t["longitude"].shape
    ny_c, nx_c = grid_c["longitude"].shape

    if "yt_ocean" not in ds.coords:
        ds = ds.assign_coords(yt_ocean=np.arange(ny_t, dtype=np.float64))
    if "xt_ocean" not in ds.coords:
        ds = ds.assign_coords(xt_ocean=np.arange(nx_t, dtype=np.float64))
    if "yu_ocean" not in ds.coords:
        ds = ds.assign_coords(yu_ocean=np.arange(ny_c, dtype=np.float64))
    if "xu_ocean" not in ds.coords:
        ds = ds.assign_coords(xu_ocean=np.arange(nx_c, dtype=np.float64))

    if "geolon_t" not in ds.variables:
        ds["geolon_t"] = xr.DataArray(
            grid_t["longitude"].values,
            dims=("yt_ocean", "xt_ocean"),
            coords={"yt_ocean": ds["yt_ocean"], "xt_ocean": ds["xt_ocean"]},
        )
    if "geolat_t" not in ds.variables:
        ds["geolat_t"] = xr.DataArray(
            grid_t["latitude"].values,
            dims=("yt_ocean", "xt_ocean"),
            coords={"yt_ocean": ds["yt_ocean"], "xt_ocean": ds["xt_ocean"]},
        )
    if "geolon_c" not in ds.variables:
        ds["geolon_c"] = xr.DataArray(
            grid_c["longitude"].values,
            dims=("yu_ocean", "xu_ocean"),
            coords={"yu_ocean": ds["yu_ocean"], "xu_ocean": ds["xu_ocean"]},
        )
    if "geolat_c" not in ds.variables:
        ds["geolat_c"] = xr.DataArray(
            grid_c["latitude"].values,
            dims=("yu_ocean", "xu_ocean"),
            coords={"yu_ocean": ds["yu_ocean"], "xu_ocean": ds["xu_ocean"]},
        )
print(ds)

## 2) Helper functions

These functions:
- pick coordinate and mask variables robustly
- classify basins from lon/lat using Natural Earth ocean basins
- map to IDs:
  - `1`: global_ocean
  - `2`: atlantic_arctic_ocean
  - `3`: indian_pacific_ocean

In [ ]:
FILL_VALUE = np.float32(-1.0e20)

def _first_existing(ds, names):
    for n in names:
        if n in ds.variables:
            return n
    raise KeyError(f"None of these variables exist: {names}")

def _get_lon_lat(ds, grid_tag):
    if grid_tag == "tt":
        lon_name = _first_existing(ds, ["geolon_t", "xt_ocean"])
        lat_name = _first_existing(ds, ["geolat_t", "yt_ocean"])
        x_name = "xt_ocean" if "xt_ocean" in ds.variables else ds[lon_name].dims[-1]
        y_name = "yt_ocean" if "yt_ocean" in ds.variables else ds[lat_name].dims[0]
    elif grid_tag == "uu":
        lon_name = _first_existing(ds, ["geolon_c", "xu_ocean"])
        lat_name = _first_existing(ds, ["geolat_c", "yu_ocean"])
        x_name = "xu_ocean" if "xu_ocean" in ds.variables else ds[lon_name].dims[-1]
        y_name = "yu_ocean" if "yu_ocean" in ds.variables else ds[lat_name].dims[0]
    elif grid_tag == "ut":
        lon_name = _first_existing(ds, ["geolon_c", "xu_ocean"])
        lat_name = _first_existing(ds, ["geolat_t", "yt_ocean"])
        x_name = "xu_ocean" if "xu_ocean" in ds.variables else ds[lon_name].dims[-1]
        y_name = "yt_ocean" if "yt_ocean" in ds.variables else ds[lat_name].dims[0]
    elif grid_tag == "tu":
        lon_name = _first_existing(ds, ["geolon_t", "xt_ocean"])
        lat_name = _first_existing(ds, ["geolat_c", "yu_ocean"])
        x_name = "xt_ocean" if "xt_ocean" in ds.variables else ds[lon_name].dims[-1]
        y_name = "yu_ocean" if "yu_ocean" in ds.variables else ds[lat_name].dims[0]
    else:
        raise ValueError(f"Unknown grid_tag: {grid_tag}")

    lon = ds[lon_name]
    lat = ds[lat_name]

    # If lon/lat are 1D, build 2D mesh.
    if lon.ndim == 1 and lat.ndim == 1:
        lon2d, lat2d = xr.broadcast(lon, lat)
        lon2d = lon2d.transpose(lat2d.dims[0], lon2d.dims[0])
        lat2d = lat2d.transpose(lat2d.dims[0], lon2d.dims[1])
    else:
        lon2d = lon
        lat2d = lat

    # Normalize dims to expected y/x names if possible.
    if lon2d.ndim != 2:
        raise ValueError(f"Expected 2D lon/lat, got lon.ndim={lon2d.ndim}, lat.ndim={lat2d.ndim}")

    dims = lon2d.dims
    rename = {}
    if dims[0] != y_name:
        rename[dims[0]] = y_name
    if dims[1] != x_name:
        rename[dims[1]] = x_name
    if rename:
        lon2d = lon2d.rename(rename)
        lat2d = lat2d.rename(rename)

    return lon2d, lat2d, y_name, x_name

def _get_wet_2d(ds, grid_tag, y_name, x_name):
    candidates = {
        "tt": ["wet", "wet_t", "mask_t", "kmt"],
        "uu": ["wet_c", "wet_u", "mask_c", "kmu", "wet"],
        "ut": ["wet_ut", "mask_ut"],
        "tu": ["wet_tu", "mask_tu"],
    }

    for v in candidates[grid_tag]:
        if v in ds.variables:
            da = ds[v]
            if da.ndim >= 2 and y_name in da.dims and x_name in da.dims:
                if da.ndim == 2:
                    return (da > 0)
                # 3D/4D case: any wet level means wet in 2D
                reduce_dims = [d for d in da.dims if d not in (y_name, x_name)]
                return (da > 0).any(dim=reduce_dims)

    # Fallback: if specific UT/TU masks do not exist, infer from TT and UU masks later.
    return None

def classify_basin_ids(lon2d, lat2d, wet2d):
    """Return basin IDs as float DataArray with NaN on land.

    ID convention used here:
    1 = global_ocean
    2 = atlantic_arctic_ocean
    3 = indian_pacific_ocean
    """
    # regionmask works best with lon in [-180, 180] for Natural Earth regions.
    lon180 = ((lon2d + 180.0) % 360.0) - 180.0

    # Try newer Natural Earth tables first, then fallback.
    basins = None
    for attr in ["natural_earth_v5_1_2", "natural_earth_v5_0_0", "natural_earth_v4_1_0"]:
        if hasattr(regionmask.defined_regions, attr):
            candidate = getattr(regionmask.defined_regions, attr)
            if hasattr(candidate, "ocean_basins_50"):
                basins = candidate.ocean_basins_50
                break
    if basins is None:
        raise RuntimeError("Could not find Natural Earth ocean basins in regionmask.")

    region_id = basins.mask(lon180, lat2d)

    names = np.asarray(basins.names)
    atl_like = np.array(["Atlantic" in n or "Arctic" in n for n in names])
    indo_like = np.array(["Indian" in n or "Pacific" in n for n in names])

    atl_ids = np.where(atl_like)[0]
    indo_ids = np.where(indo_like)[0]

    atl_mask = xr.zeros_like(region_id, dtype=bool)
    for i in atl_ids:
        atl_mask = atl_mask | (region_id == i)

    indo_mask = xr.zeros_like(region_id, dtype=bool)
    for i in indo_ids:
        indo_mask = indo_mask | (region_id == i)

    # Start with global ocean (1), land as NaN.
    out = xr.where(wet2d, 1.0, np.nan)
    out = xr.where(wet2d & atl_mask, 2.0, out)
    out = xr.where(wet2d & indo_mask, 3.0, out)

    # Fallback for any unclassified wet cells (rare coastal/polar slivers).
    unclassified = wet2d & ~(atl_mask | indo_mask)
    # Simple split that usually places most of the Indo-Pacific correctly.
    indo_fallback = (lon180 >= 20.0) & (lon180 <= 150.0)
    out = xr.where(unclassified & indo_fallback, 3.0, out)
    out = xr.where(unclassified & ~indo_fallback, 2.0, out)

    return out.astype(np.float32)

def build_wet3d_from_k(ds, wet2d, y_name, x_name, st_coord, k_candidates):
    nz = st_coord.size
    lev_index = xr.DataArray(np.arange(1, nz + 1), dims=("st_ocean",), coords={"st_ocean": st_coord})

    for kname in k_candidates:
        if kname in ds.variables:
            k = ds[kname]
            if y_name in k.dims and x_name in k.dims:
                k2d = k
                if k.ndim > 2:
                    reduce_dims = [d for d in k.dims if d not in (y_name, x_name)]
                    k2d = k.max(dim=reduce_dims)
                k2d = k2d.astype(np.int32)
                wet3d = lev_index <= k2d
                return wet3d.transpose("st_ocean", y_name, x_name)

    # Fallback: repeat 2D wet mask through depth
    return wet2d.expand_dims(st_ocean=st_coord).transpose("st_ocean", y_name, x_name)

def make_mask_var(basin2d, wet3d, var_name, long_name, coords_attr, standard_name=None):
    basin3d = basin2d.expand_dims(st_ocean=wet3d["st_ocean"]).transpose(*wet3d.dims)
    out = xr.where(wet3d, basin3d, FILL_VALUE).astype(np.float32)
    out.name = var_name
    out.attrs.update({
        "long_name": long_name,
        "units": "none",
        "valid_range": np.array([0.0, 99.0], dtype=np.float32),
        "missing_value": np.float32(FILL_VALUE),
        "_FillValue": np.float32(FILL_VALUE),
        "coordinates": coords_attr,
        "cell_methods": "time: mean",
    })
    if standard_name is not None:
        out.attrs["standard_name"] = standard_name
    return out

## 3) Build 2D basin IDs on each grid

If UT/TU wet masks are missing, we infer them from TT and UU overlap.

In [ ]:
# Coordinates
lon_tt, lat_tt, y_tt, x_tt = _get_lon_lat(ds, "tt")
lon_uu, lat_uu, y_uu, x_uu = _get_lon_lat(ds, "uu")
lon_ut, lat_ut, y_ut, x_ut = _get_lon_lat(ds, "ut")
lon_tu, lat_tu, y_tu, x_tu = _get_lon_lat(ds, "tu")

# 2D wet masks
wet_tt = _get_wet_2d(ds, "tt", y_tt, x_tt)
wet_uu = _get_wet_2d(ds, "uu", y_uu, x_uu)
wet_ut = _get_wet_2d(ds, "ut", y_ut, x_ut)
wet_tu = _get_wet_2d(ds, "tu", y_tu, x_tu)

if wet_tt is None:
    raise RuntimeError("Could not determine wet mask for TT grid. Provide wet/kmt in input file.")
if wet_uu is None:
    # Conservative fallback for UU when dedicated mask is absent
    wet_uu = wet_tt.rename({y_tt: y_uu, x_tt: x_uu}) if (wet_tt.shape == lon_uu.shape) else xr.ones_like(lon_uu, dtype=bool)

if wet_ut is None:
    wet_ut = (wet_tt.rename({y_tt: y_ut, x_tt: x_ut}) & wet_uu.rename({y_uu: y_ut, x_uu: x_ut}))
if wet_tu is None:
    wet_tu = (wet_tt.rename({y_tt: y_tu, x_tt: x_tu}) & wet_uu.rename({y_uu: y_tu, x_uu: x_tu}))

# Basin IDs
basin_tt = classify_basin_ids(lon_tt, lat_tt, wet_tt)
basin_uu = classify_basin_ids(lon_uu, lat_uu, wet_uu)
basin_ut = classify_basin_ids(lon_ut, lat_ut, wet_ut)
basin_tu = classify_basin_ids(lon_tu, lat_tu, wet_tu)

print("TT basin IDs:", np.unique(basin_tt.values[~np.isnan(basin_tt.values)]))
print("UU basin IDs:", np.unique(basin_uu.values[~np.isnan(basin_uu.values)]))

## 4) Build 3D masks

If `st_ocean` is present, we use it directly. Otherwise we create `1..DEFAULT_NZ`.

Vertical wetness priority:
- TT: `kmt` (fallback: repeat 2D wet mask)
- UU: `kmu` then `kmt` (fallback: repeat 2D wet mask)
- UT/TU: `kmt` (fallback: repeat 2D wet mask)

In [ ]:
if "st_ocean" in ds.variables:
    st_ocean = ds["st_ocean"]
else:
    st_ocean = xr.DataArray(np.arange(1, DEFAULT_NZ + 1, dtype=np.float64), dims=("st_ocean",), name="st_ocean")
    st_ocean.attrs.update({"long_name": "depth level index", "units": "1", "positive": "down"})

wet3d_tt = build_wet3d_from_k(ds, wet_tt, y_tt, x_tt, st_ocean, ["kmt"])
wet3d_uu = build_wet3d_from_k(ds, wet_uu, y_uu, x_uu, st_ocean, ["kmu", "kmt"])
wet3d_ut = build_wet3d_from_k(ds, wet_ut, y_ut, x_ut, st_ocean, ["kmt"])
wet3d_tu = build_wet3d_from_k(ds, wet_tu, y_tu, x_tu, st_ocean, ["kmt"])

mask_ttcell = make_mask_var(
    basin_tt, wet3d_tt, "mask_ttcell", "ttcell basin mask", "geolon_t geolat_t"
)
mask_uucell = make_mask_var(
    basin_uu, wet3d_uu, "mask_uucell", "uucell basin mask", "geolon_c geolat_c",
    standard_name="sea_water_x_velocity"
)
mask_utcell = make_mask_var(
    basin_ut, wet3d_ut, "mask_utcell", "utcell basin mask", "geolon_c geolat_t",
    standard_name="ocean_x_mass_transport"
)
mask_tucell = make_mask_var(
    basin_tu, wet3d_tu, "mask_tucell", "tucell basin mask", "geolon_t geolat_c",
    standard_name="ocean_y_mass_transport"
)

## 5) Assemble and save NetCDF

This writes a file with dimensions analogous to your reference file.

In [ ]:
def _coord_or_index(ds, name, size):
    if name in ds.variables:
        return ds[name]
    return xr.DataArray(np.arange(size, dtype=np.float64), dims=(name,), name=name)

yt_ocean = _coord_or_index(ds, "yt_ocean", mask_ttcell.sizes[y_tt])
xt_ocean = _coord_or_index(ds, "xt_ocean", mask_ttcell.sizes[x_tt])
yu_ocean = _coord_or_index(ds, "yu_ocean", mask_uucell.sizes[y_uu])
xu_ocean = _coord_or_index(ds, "xu_ocean", mask_uucell.sizes[x_uu])

mask_ttcell = mask_ttcell.rename({y_tt: "yt_ocean", x_tt: "xt_ocean"})
mask_uucell = mask_uucell.rename({y_uu: "yu_ocean", x_uu: "xu_ocean"})
mask_utcell = mask_utcell.rename({y_ut: "yt_ocean", x_ut: "xu_ocean"})
mask_tucell = mask_tucell.rename({y_tu: "yu_ocean", x_tu: "xt_ocean"})

ds_out = xr.Dataset(
    data_vars={
        "mask_ttcell": mask_ttcell,
        "mask_uucell": mask_uucell,
        "mask_utcell": mask_utcell,
        "mask_tucell": mask_tucell,
    },
    coords={
        "st_ocean": st_ocean,
        "yt_ocean": yt_ocean,
        "xt_ocean": xt_ocean,
        "yu_ocean": yu_ocean,
        "xu_ocean": xu_ocean,
    },
)

# Keep simple encoding and preserve fill values.
encoding = {
    name: {"_FillValue": np.float32(FILL_VALUE), "dtype": "float32"}
    for name in ["mask_ttcell", "mask_uucell", "mask_utcell", "mask_tucell"]
}

ds_out.to_netcdf(OUTPUT_FILE, encoding=encoding)
print(f"Wrote: {OUTPUT_FILE}")
print(ds_out)

## 6) Quick sanity checks

Check counts per basin code and confirm expected dimensions.

In [ ]:
def counts(arr):
    valid = arr.values[arr.values > 0]
    vals, cnt = np.unique(valid.astype(np.int32), return_counts=True)
    return dict(zip(vals.tolist(), cnt.tolist()))

print("mask_ttcell counts:", counts(ds_out["mask_ttcell"].isel(st_ocean=0)))
print("mask_uucell counts:", counts(ds_out["mask_uucell"].isel(st_ocean=0)))
print("mask_utcell counts:", counts(ds_out["mask_utcell"].isel(st_ocean=0)))
print("mask_tucell counts:", counts(ds_out["mask_tucell"].isel(st_ocean=0)))

## Notes on matching your legacy mask exactly

- This notebook reproduces the same mask *structure* and a practical basin coding workflow.
- Exact cell-by-cell match can differ if your legacy file used custom strait gates or a different polygon source.
- If you have a MOM6 basin function already, you can drop that logic into `classify_basin_ids` and keep the MOM5 grid handling from this notebook unchanged.